In [1]:
# ── Ячейка 1: импорты и инфраструктура ────────────────────────────────────────
import sys; sys.path.insert(0, "..")
import gc, time
import numpy as np
import joblib

from src import config, data, evaluate

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con)   # assert: 76 признаков, таргет не утёк

n_train = data.count_rows(con, config.TRAIN_PATH)
n_val   = data.count_rows(con, config.VAL_PATH)
n_test  = data.count_rows(con, config.TEST_PATH)

# scale_pos_weight = n_neg / n_pos (компенсация дисбаланса для XGBoost).
counts = con.execute(
    f"SELECT label_binary, COUNT(*) FROM read_parquet('{config.TRAIN_PATH}') "
    "GROUP BY label_binary ORDER BY label_binary"
).fetchall()
n_neg, n_pos = counts[0][1], counts[1][1]
scale_pos_w = n_neg / n_pos

print(f"Признаков (tree-view): {len(feature_cols)}")
print(f"train={n_train:,}  val={n_val:,}  test={n_test:,}")
print(f"scale_pos_weight = {n_neg:,}/{n_pos:,} = {scale_pos_w:.4f}")

Признаков (tree-view): 76
train=9,583,883  val=2,053,688  test=2,053,697
scale_pos_weight = 7,000,000/2,583,883 = 2.7091


In [2]:
# ── Ячейка 2: загрузка train и val в RAM (tree-view, сырьё) ───────────────────
# train — для обучения; val — для eval_set (early stopping во время обучения).
# Оба сырые 76 признаков, без препроцессора (деревья инвариантны к масштабу).
def _load(path):
    Xc, yc = [], []
    for X_b, y_b in data.iter_batches(path, feature_cols):
        Xc.append(X_b); yc.append(y_b)
    X = np.concatenate(Xc); y = np.concatenate(yc)
    del Xc, yc; gc.collect()
    return X, y

print("Загружаем train...")
t0 = time.time()
X_train, y_train = _load(config.TRAIN_PATH)
print(f"  X_train: {X_train.shape}  {X_train.nbytes/1024**3:.2f} GB  {time.time()-t0:.1f}s")

print("Загружаем val (для eval_set)...")
X_val, y_val = _load(config.VAL_PATH)
print(f"  X_val:   {X_val.shape}  {X_val.nbytes/1024**3:.2f} GB")

Загружаем train...
  X_train: (9583883, 76)  2.71 GB  4.4s
Загружаем val (для eval_set)...
  X_val:   (2053688, 76)  0.58 GB


In [3]:
# ── Ячейка 3: обучение XGBoost с early stopping ───────────────────────────────
# eval_set=val → честная ранняя остановка: val используется ПО НАЗНАЧЕНИЮ
# (выбор числа деревьев), а не как вторая test. early_stopping_rounds=20.
# Ставим запас n_estimators=300, чтобы early stopping сам нашёл оптимум,
# а не упёрся в потолок (как было в старом ноутбуке с n_estimators=100).
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_w,
    tree_method="hist",
    n_jobs=-1,
    random_state=config.RANDOM_STATE,
    eval_metric="aucpr",
    early_stopping_rounds=20,
    verbosity=1,
)

print("Обучение XGBoost (max n_estimators=300, early_stopping=20)...")
t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=20)
print(f"\nГотово за {time.time()-t0:.1f}s")
print(f"Лучшая итерация: {model.best_iteration}  |  "
      f"val PR-AUC: {model.best_score:.6f}")
print(f"Early stopping сработал: {model.best_iteration < 299}")

del X_train, y_train, X_val, y_val; gc.collect()

joblib.dump(model, config.MODELS_DIR / "xgb.pkl")
print(f"Модель сохранена → {config.MODELS_DIR / 'xgb.pkl'}")

Обучение XGBoost (max n_estimators=300, early_stopping=20)...
[0]	validation_0-aucpr:0.99891
[20]	validation_0-aucpr:0.99928
[40]	validation_0-aucpr:0.99983
[60]	validation_0-aucpr:0.99993
[80]	validation_0-aucpr:0.99993
[100]	validation_0-aucpr:0.99993
[120]	validation_0-aucpr:0.99993
[140]	validation_0-aucpr:0.99994
[160]	validation_0-aucpr:0.99994
[180]	validation_0-aucpr:0.99994
[200]	validation_0-aucpr:0.99994
[208]	validation_0-aucpr:0.99994

Готово за 187.3s
Лучшая итерация: 188  |  val PR-AUC: 0.999936
Early stopping сработал: True
Модель сохранена → /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/models/xgb.pkl


In [4]:
# ── Ячейка 4: оценка на VAL (подбор порога) ───────────────────────────────────
# tree-view: предикт на сыром батче, БЕЗ препроцессора.
# XGBoost по умолчанию использует best_iteration для предсказаний.
def predict_xgb(X_raw):
    return model.predict_proba(X_raw)[:, 1]

m_val = evaluate.evaluate(
    "xgb", predict_xgb, config.VAL_PATH, feature_cols,
    threshold=None, split_name="val",
)
best_thr = m_val["threshold"]
print(f"\nПодобранный на val порог: {best_thr}")


  xgb  (val)
строк 2,053,688 за 2.31s  |  порог 0.7613 (tuned_on_this_split)
F1_macro 0.9988  MCC 0.9976  PR-AUC 0.999936  ROC-AUC 0.999977  FPR@95 8.6e-05
TN=1,498,081  FP=1,919  FN=6  TP=553,682  (FPR=0.0013, FNR=0.0000)

per-class recall:
  Benign                           n=1,500,000  recall=99.87%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,399  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,832  recall=100.00%
  Bot                              n=  14,423  recall=99.99%
  SSH-Patator                      n=  13,897  recall=100.00%
  DoS GoldenEye                    n=   4,029  recall=100.00%
  DoS Slowloris                    n=   1,541  recall=99.94%
  DDoS LOIC-UDP                    n=     357  recall=100.00%
  Web Attack - Brute Force         n=      39  recall=100.00%
  Web Attack - 

In [5]:
# ── Ячейка 5: оценка на TEST (замороженный порог) ─────────────────────────────
m_test = evaluate.evaluate(
    "xgb", predict_xgb, config.TEST_PATH, feature_cols,
    threshold=best_thr, split_name="test",
)


  xgb  (test)
строк 2,053,697 за 2.34s  |  порог 0.7613 (frozen_from_val)
F1_macro 0.9985  MCC 0.997  PR-AUC 0.999389  ROC-AUC 0.999832  FPR@95 0.000432
TN=1,497,564  FP=2,436  FN=5  TP=553,692  (FPR=0.0016, FNR=0.0000)

per-class recall:
  Benign                           n=1,500,000  recall=99.84%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,400  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,833  recall=100.00%
  Bot                              n=  14,424  recall=99.99%
  SSH-Patator                      n=  13,898  recall=100.00%
  DoS GoldenEye                    n=   4,030  recall=100.00%
  DoS Slowloris                    n=   1,542  recall=99.94%
  DDoS LOIC-UDP                    n=     358  recall=100.00%
  Web Attack - Brute Force         n=      39  recall=97.44%
  Web Attack - XSS 

In [6]:
# ── Ячейка 6: feature importance (для записки) ────────────────────────────────
import pandas as pd

fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Топ-15 признаков по важности (XGBoost):")
print(fi.head(15).to_string(index=False))
print(f"\nТоп-3 суммарно: {fi['importance'].head(3).sum():.4f}")

Топ-15 признаков по важности (XGBoost):
               feature  importance
fwd_tcp_init_win_bytes    0.629593
   bwd_pkt_hdr_len_tot    0.152131
         down_up_ratio    0.095796
bwd_tcp_init_win_bytes    0.067044
           bwd_pkt_cnt    0.026643
   fwd_bulk_bytes_mean    0.004980
             idle_mean    0.004959
       bwd_pkt_len_tot    0.003360
              flag_SYN    0.002380
       fwd_pkt_len_tot    0.002350
         bwd_pkt_per_s    0.001733
              flag_ece    0.001172
          fwd_flag_psh    0.000996
       fwd_pkt_len_max    0.000805
   fwd_pkt_hdr_len_tot    0.000733

Топ-3 суммарно: 0.8775
